# 02 Time-based split and baseline model

A simple, leak-proof baseline to measure every later improvement against.

**Setup**
- Train: 2019 (924,850 transactions, 0.564% fraud). Validation: 2020-01-01 to 2020-06-21 (371,825 transactions, 0.615% fraud).
- Split by time, not randomly, so the model is always evaluated on transactions after the ones it learned from.
- Logistic regression on amount (log-scaled), category, and hour (one-hot), with preprocessing inside a scikit-learn pipeline.

**Findings**
- PR-AUC 0.438, against about 0.006 for a random model.
- At the default 0.5 threshold, precision is 0.809 but recall only 0.252: the model catches a quarter of fraud.
- Accuracy is 99.5%, barely above the 99.4% of always predicting "genuine", which shows why accuracy is useless here.
- Lowering the threshold to 0.05 raises recall to 0.662 but drops precision to 0.240. Each extra fraud caught costs more false alarms (from about 1 to about 9), so choosing the threshold is a separate business decision.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/fraudTrain.csv", index_col=0,
                 parse_dates=["trans_date_trans_time"])

cutoff = pd.Timestamp("2020-01-01")
tr  = df[df["trans_date_trans_time"] <  cutoff].copy()
val = df[df["trans_date_trans_time"] >= cutoff].copy()

for d in (tr, val):
    d["hour"] = d["trans_date_trans_time"].dt.hour

for name, part in [("train", tr), ("val", val)]:
    t = part["trans_date_trans_time"]
    print(f"{name}: {len(part):,} rows | fraud {part['is_fraud'].mean():.3%} | {t.min().date()} → {t.max().date()}")

train: 924,850 rows | fraud 0.564% | 2019-01-01 → 2019-12-31
val: 371,825 rows | fraud 0.615% | 2020-01-01 → 2020-06-21


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report

features = ["amt", "category", "hour"]
X_tr, y_tr   = tr[features],  tr["is_fraud"]
X_val, y_val = val[features], val["is_fraud"]

preprocess = ColumnTransformer([
    ("amount", Pipeline([("log", FunctionTransformer(np.log1p)),
                         ("scale", StandardScaler())]), ["amt"]),
    ("cats", OneHotEncoder(handle_unknown="ignore"), ["category", "hour"]),
])

model = Pipeline([("prep", preprocess),
                  ("clf", LogisticRegression(max_iter=1000))])

model.fit(X_tr, y_tr)
val_scores = model.predict_proba(X_val)[:, 1]

print(f"PR-AUC: {average_precision_score(y_val, val_scores):.3f}  (random ≈ {y_val.mean():.3f})")
print(classification_report(y_val, (val_scores >= 0.5).astype(int), digits=3))

PR-AUC: 0.438  (random ≈ 0.006)
              precision    recall  f1-score   support

           0      0.995     1.000     0.998    369539
           1      0.809     0.252     0.384      2286

    accuracy                          0.995    371825
   macro avg      0.902     0.626     0.691    371825
weighted avg      0.994     0.995     0.994    371825



In [3]:
from sklearn.metrics import precision_score, recall_score

for t in [0.5, 0.3, 0.1, 0.05]:
    pred = (val_scores >= t).astype(int)
    print(f"threshold {t:.2f}: precision {precision_score(y_val, pred):.3f} | "
          f"recall {recall_score(y_val, pred):.3f} | flags {pred.sum():,}")

threshold 0.50: precision 0.809 | recall 0.252 | flags 711
threshold 0.30: precision 0.691 | recall 0.365 | flags 1,209
threshold 0.10: precision 0.451 | recall 0.504 | flags 2,559
threshold 0.05: precision 0.240 | recall 0.662 | flags 6,301
